# CLINC/OOS Intent Classification Lab with DistilBERT and PEFT

This notebook is a classroom-friendly version of the intent classification lab.
It uses a compact encoder model, direct sequence classification, and a simpler debugging path than a generative LLM pipeline.


## 1. Learning Objectives

- Build a strong lexical baseline with TF-IDF and LinearSVC.
- Fine-tune DistilBERT for 150-way in-scope intent classification.
- Use PEFT LoRA on a small encoder model instead of a large decoder model.
- Compare classical and transformer-based baselines under the same split and metrics.

Important notes:
- The notebook assumes the raw CLINC/OOS files already exist in `./raw`.
- OOS files are loaded for inspection, but the core lab remains an in-scope classification task.


In [1]:
# Uncomment this only if your environment is missing the required packages.
# Keep the setup minimal for classroom use.

#!pip -q install datasets transformers peft accelerate scikit-learn pandas numpy


In [2]:
from __future__ import annotations

import json
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


/home/dryjins/miniconda3/envs/lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).


In [3]:
def set_seed(seed: int = 42) -> None:
    """Set all random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


## 2. Raw File Layout

The notebook expects the same raw file structure used in the earlier lab.
This keeps the data split consistent while simplifying the model side of the pipeline.


In [4]:
RAW_DIR = Path("./raw")
DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

IN_SCOPE_FILES = {
    "train": RAW_DIR / "is_train.json",
    "val": RAW_DIR / "is_val.json",
    "test": RAW_DIR / "is_test.json",
}
OOS_FILES = {
    "train": RAW_DIR / "oos_train.json",
    "val": RAW_DIR / "oos_val.json",
    "test": RAW_DIR / "oos_test.json",
}

for split_name, path in {**IN_SCOPE_FILES, **{f"oos_{k}": v for k, v in OOS_FILES.items()}}.items():
    print(split_name, path, path.exists())


train raw/is_train.json True
val raw/is_val.json True
test raw/is_test.json True
oos_train raw/oos_train.json True
oos_val raw/oos_val.json True
oos_test raw/oos_test.json True


## 3. Inspect One Raw File

Inspect the real schema once before trusting any loader.
This reduces the chance of silent format mismatch bugs.


In [5]:
with IN_SCOPE_FILES["train"].open("r", encoding="utf-8") as f:
    train_raw_preview = json.load(f)

print(type(train_raw_preview))
if isinstance(train_raw_preview, list):
    print("num_examples:", len(train_raw_preview))
    print("first_example:", train_raw_preview[0])
elif isinstance(train_raw_preview, dict):
    print("keys:", list(train_raw_preview.keys())[:20])


<class 'list'>
num_examples: 15000
first_example: ['what expression would i use to say i love you if i were an italian', 'translate']


## 4. Data Loader

The helper below supports the common CLINC-style list format and a few common dictionary-based variants.
If your mirror uses a different schema, edit only this cell.


In [6]:
def load_split(path: Path) -> pd.DataFrame:
    """Load one split from a raw CLINC/OOS JSON file."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        if "data" in data and isinstance(data["data"], list):
            data = data["data"]
        elif "records" in data and isinstance(data["records"], list):
            data = data["records"]
        else:
            raise ValueError(f"Unsupported dict schema in {path}")

    rows = []
    for item in data:
        if isinstance(item, list) and len(item) >= 2:
            text, intent = item[0], item[1]
        elif isinstance(item, dict):
            text = item.get("text") or item.get("utterance") or item.get("sentence")
            intent = item.get("intent") or item.get("label")
        else:
            raise ValueError(f"Unsupported item schema in {path}: {type(item)}")

        if text is None or intent is None:
            continue

        rows.append({"text": str(text), "intent": str(intent)})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"Loaded an empty dataframe from {path}")

    return df


In [7]:
train_df = load_split(IN_SCOPE_FILES["train"])
val_df = load_split(IN_SCOPE_FILES["val"])
test_df = load_split(IN_SCOPE_FILES["test"])

oos_train_df = load_split(OOS_FILES["train"]) if OOS_FILES["train"].exists() else pd.DataFrame(columns=["text", "intent"])
oos_val_df = load_split(OOS_FILES["val"]) if OOS_FILES["val"].exists() else pd.DataFrame(columns=["text", "intent"])
oos_test_df = load_split(OOS_FILES["test"]) if OOS_FILES["test"].exists() else pd.DataFrame(columns=["text", "intent"])

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("oos_train:", oos_train_df.shape)
print("oos_val:", oos_val_df.shape)
print("oos_test:", oos_test_df.shape)


train: (15000, 2)
val: (3000, 2)
test: (4500, 2)
oos_train: (100, 2)
oos_val: (100, 2)
oos_test: (1000, 2)


## 5. Label Encoding

The training objective uses only the in-scope labels.
The classifier therefore predicts exactly one of the 150 in-scope intents.


In [8]:
label_list = sorted(train_df["intent"].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

for df in [train_df, val_df, test_df]:
    df["label_id"] = df["intent"].map(label2id)

assert train_df["label_id"].isna().sum() == 0
assert val_df["label_id"].isna().sum() == 0
assert test_df["label_id"].isna().sum() == 0

print("num_labels:", len(label_list))
print(pd.DataFrame(label_list, columns=["label"]).head(20).to_markdown())


num_labels: 150
|    | label               |
|---:|:--------------------|
|  0 | accept_reservations |
|  1 | account_blocked     |
|  2 | alarm               |
|  3 | application_status  |
|  4 | apr                 |
|  5 | are_you_a_bot       |
|  6 | balance             |
|  7 | bill_balance        |
|  8 | bill_due            |
|  9 | book_flight         |
| 10 | book_hotel          |
| 11 | calculator          |
| 12 | calendar            |
| 13 | calendar_update     |
| 14 | calories            |
| 15 | cancel              |
| 16 | cancel_reservation  |
| 17 | car_rental          |
| 18 | card_declined       |
| 19 | carry_on            |


## 6. Optional Downsampling

Classroom labs do not always need the full dataset.
Use the subset switch below if you need a faster iteration cycle.


In [9]:
USE_SUBSET = False
TRAIN_SAMPLES = 6000
VAL_SAMPLES = 1500
TEST_SAMPLES = 1500

if USE_SUBSET:
    train_local_df = train_df.sample(min(TRAIN_SAMPLES, len(train_df)), random_state=42).reset_index(drop=True)
    val_local_df = val_df.sample(min(VAL_SAMPLES, len(val_df)), random_state=42).reset_index(drop=True)
    test_local_df = test_df.sample(min(TEST_SAMPLES, len(test_df)), random_state=42).reset_index(drop=True)
else:
    train_local_df = train_df.reset_index(drop=True)
    val_local_df = val_df.reset_index(drop=True)
    test_local_df = test_df.reset_index(drop=True)

print(train_local_df.shape, val_local_df.shape, test_local_df.shape)


(15000, 3) (3000, 3) (4500, 3)


## 7. Sparse Baseline

A classical sparse baseline is still important in intent classification.
It provides a sanity check and often performs surprisingly well on lexical tasks.


In [10]:
svm_clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
    ("clf", LinearSVC()),
])

svm_clf.fit(train_local_df["text"], train_local_df["intent"])
svm_pred = svm_clf.predict(test_local_df["text"])
svm_acc = accuracy_score(test_local_df["intent"], svm_pred)
svm_macro_f1 = f1_score(test_local_df["intent"], svm_pred, average="macro")

print("TF-IDF + LinearSVC accuracy:", round(svm_acc, 4))
print("TF-IDF + LinearSVC macro F1:", round(svm_macro_f1, 4))


TF-IDF + LinearSVC accuracy: 0.9111
TF-IDF + LinearSVC macro F1: 0.9101


## 8. DistilBERT Backbone

This version uses `distilbert-base-uncased` instead of `bert-tiny`.
That change increases capacity and usually leads to a more realistic classroom baseline for 150-way intent classification.


In [11]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

if hasattr(base_model.config, "use_cache"):
    base_model.config.use_cache = False

print(base_model.__class__.__name__)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification


In [12]:
# Inspect module names before applying LoRA.
# DistilBERT typically uses q_lin, k_lin, v_lin, and out_lin.
for name, module in base_model.named_modules():
    if any(key in name for key in ["q_lin", "k_lin", "v_lin", "out_lin", "classifier", "pre_classifier"]):
        print(name)


distilbert.transformer.layer.0.attention.q_lin
distilbert.transformer.layer.0.attention.k_lin
distilbert.transformer.layer.0.attention.v_lin
distilbert.transformer.layer.0.attention.out_lin
distilbert.transformer.layer.1.attention.q_lin
distilbert.transformer.layer.1.attention.k_lin
distilbert.transformer.layer.1.attention.v_lin
distilbert.transformer.layer.1.attention.out_lin
distilbert.transformer.layer.2.attention.q_lin
distilbert.transformer.layer.2.attention.k_lin
distilbert.transformer.layer.2.attention.v_lin
distilbert.transformer.layer.2.attention.out_lin
distilbert.transformer.layer.3.attention.q_lin
distilbert.transformer.layer.3.attention.k_lin
distilbert.transformer.layer.3.attention.v_lin
distilbert.transformer.layer.3.attention.out_lin
distilbert.transformer.layer.4.attention.q_lin
distilbert.transformer.layer.4.attention.k_lin
distilbert.transformer.layer.4.attention.v_lin
distilbert.transformer.layer.4.attention.out_lin
distilbert.transformer.layer.5.attention.q_lin
dis

## 9. Apply PEFT LoRA

The target modules below are aligned to the DistilBERT attention block naming.
This fixes the common mistake of using BERT-style `query` and `value` names on a DistilBERT backbone.


In [13]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 853,398 || all params: 67,922,220 || trainable%: 1.2564


## 10. Tokenization and Dataset Conversion

The task is direct sequence classification, so every text maps to exactly one label id.
There is no prompt construction step and no label-string parser.


In [14]:
MAX_LENGTH = 64


def tokenize_function(batch: Dict[str, List[str]]) -> Dict[str, List[int]]:
    """Tokenize a batch of examples for sequence classification."""
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


train_ds = Dataset.from_pandas(train_local_df[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)
val_ds = Dataset.from_pandas(val_local_df[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)
test_ds = Dataset.from_pandas(test_local_df[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

keep_columns = ["input_ids", "attention_mask", "labels"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_columns])
val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_columns])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_columns])

collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map: 100%|█████████████████████████████████████████████████████████████████████████████| 4500/4500 [00:00<00:00, 20390.66 examples/s]


## 11. Metrics

Accuracy is intuitive, while macro-F1 is more honest for a many-class task.
Both metrics are retained to keep the comparison fair.


In [15]:
def compute_metrics(eval_pred):
    """Compute accuracy and macro-F1 from classifier logits."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


## 12. Train the PEFT Model

The hyperparameters below are conservative and should be reasonable on a small GPU.
If you run out of memory, reduce the batch size before changing the rest of the setup.


In [17]:
training_args = TrainingArguments(
    output_dir="output/distilbert_peft_clinc",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_stats = trainer.train()
trainer_stats


/tmp/ipykernel_7457/3707089981.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.496500,0.439356,0.881667,0.878753
2,0.220600,0.265436,0.934000,0.933314
3,0.178000,0.219807,0.944333,0.944197
4,0.116600,0.201348,0.955333,0.955024
5,0.106300,0.194310,0.956000,0.955902


TrainOutput(global_step=4690, training_loss=0.4801608144601525, metrics={'train_runtime': 94.7453, 'train_samples_per_second': 791.596, 'train_steps_per_second': 49.501, 'total_flos': 363462362352000.0, 'train_loss': 0.4801608144601525, 'epoch': 5.0})

## 13. Test Evaluation

The classifier predicts one of the 150 in-scope labels by `argmax` over logits.
If predictions look strange, the model is making a wrong choice inside the label set rather than generating an out-of-set string.


In [18]:
test_output = trainer.predict(test_ds)
bert_pred_ids = np.argmax(test_output.predictions, axis=-1)
bert_pred_labels = [id2label[i] for i in bert_pred_ids]

bert_acc = accuracy_score(test_local_df["label_id"], bert_pred_ids)
bert_macro_f1 = f1_score(test_local_df["label_id"], bert_pred_ids, average="macro")

print("DistilBERT + PEFT accuracy:", round(bert_acc, 4))
print("DistilBERT + PEFT macro F1:", round(bert_macro_f1, 4))


DistilBERT + PEFT accuracy: 0.9529
DistilBERT + PEFT macro F1: 0.9528


## 14. Comparison Table

The main classroom comparison is between a sparse linear baseline and a compact transformer with PEFT.
This makes it easier to discuss what model capacity buys and what it does not.


In [19]:
results_df = pd.DataFrame(
    [
        {"model": "TF-IDF + LinearSVC", "accuracy": svm_acc, "macro_f1": svm_macro_f1},
        {"model": "DistilBERT + PEFT", "accuracy": bert_acc, "macro_f1": bert_macro_f1},
    ]
)
results_df


,model,accuracy,macro_f1
0,TF-IDF + LinearSVC,0.911111,0.910092
1,DistilBERT + PEFT,0.952889,0.952754


## 15. Lightweight Error Analysis

Inspecting failures is often more informative than inspecting one aggregate score.
This cell makes it easy to see which intents are still confused.


In [20]:
error_df = test_local_df.copy()
error_df["pred_intent"] = bert_pred_labels
error_df["correct"] = error_df["intent"] == error_df["pred_intent"]

error_df[~error_df["correct"]].head(20)


,text,intent,label_id,pred_intent,correct
3,how do you say fast in spanish,translate,131,change_language,False
53,repeat what the weather will be like,transfer,130,weather,False
65,who set up the numbers for it,timer,122,who_made_you,False
88,let me know when it's been 5 minutes,timer,122,time,False
112,i heard some woman say she was going to yerd m...,definition,34,repeat,False
148,what's the answer to it all,meaning_of_life,68,maybe,False
172,how do i sign up for a new allstatedplan,insurance_change,58,new_card,False
179,phone number to state farm to change insurance,insurance_change,58,make_call,False
258,is it ok if i use some of my pto on may 24th t...,pto_request,89,pto_request_status,False
268,vacation request information,pto_request,89,pto_request_status,False
